
# Baseline Evaluation: Random Forest and Ridge Regression

This notebook evaluates two baseline machine learning models for drug synergy prediction:

- **Random Forest Regressor**
- **Ridge Regression**

The input data is a concatenated feature vector of:
- Drug A (ECFP6 + physicochem + tox)
- Drug B (ECFP6 + physicochem + tox)
- Cell Line (Gene expression)

We use the exact same 5-fold data split as in the DeepSynergy publication (Preuer et al., 2018), and compare our results to their reported performance.


In [ ]:

import pickle
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error


## Load Input Data (X.p and labels.csv)

In [ ]:

# Load features
with open("X.p", "rb") as f:
    X = pickle.load(f)
print(f"X shape: {X.shape}")

# Load labels
labels = pd.read_csv("labels.csv", index_col=0)
labels = pd.concat([labels, labels])  # For AB and BA pairs
folds = labels['fold'].values
synergy_scores = labels['synergy'].values
print(f"Labels loaded: {len(synergy_scores)} samples")


## Define Hyperparameters and Initialize

In [ ]:

param_grid_rf = {
    'n_estimators': [128, 512, 1024],
    'max_features': ['sqrt', 256]
}

param_grid_ridge = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

rf_mse_scores = []
ridge_mse_scores = []


## Cross-Validation Loop (5 Folds)

In [ ]:

for fold in range(5):
    print(f"--- Fold {fold+1} ---")
    start_time = time.time()

    train_idx = np.where(folds != fold)[0]
    test_idx = np.where(folds == fold)[0]

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = synergy_scores[train_idx], synergy_scores[test_idx]

    # Random Forest
    rf = RandomForestRegressor(random_state=0)
    rf_grid = GridSearchCV(rf, param_grid_rf, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    rf_grid.fit(X_train, y_train)
    y_pred_rf = rf_grid.best_estimator_.predict(X_test)
    rf_mse = mean_squared_error(y_test, y_pred_rf)
    rf_mse_scores.append(rf_mse)
    print(f"RF MSE: {rf_mse:.2f}, Params: {rf_grid.best_params_}")

    # Ridge Regression
    ridge = Ridge()
    ridge_grid = GridSearchCV(ridge, param_grid_ridge, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    ridge_grid.fit(X_train, y_train)
    y_pred_ridge = ridge_grid.best_estimator_.predict(X_test)
    ridge_mse = mean_squared_error(y_test, y_pred_ridge)
    ridge_mse_scores.append(ridge_mse)
    print(f"Ridge MSE: {ridge_mse:.2f}, Alpha: {ridge_grid.best_params_['alpha']}")

    print(f"Fold {fold+1} done in {(time.time() - start_time)/60:.2f} min")


## Final Evaluation and Visualization

In [ ]:

print("Random Forest MSEs:", rf_mse_scores)
print(f"Mean RF MSE: {np.mean(rf_mse_scores):.2f}")
print("Ridge Regression MSEs:", ridge_mse_scores)
print(f"Mean Ridge MSE: {np.mean(ridge_mse_scores):.2f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, 6), rf_mse_scores, marker='o', label='Random Forest')
plt.plot(range(1, 6), ridge_mse_scores, marker='o', label='Ridge Regression')
plt.axhline(np.mean(rf_mse_scores), linestyle='--', color='blue', label=f'RF Mean: {np.mean(rf_mse_scores):.1f}')
plt.axhline(np.mean(ridge_mse_scores), linestyle='--', color='orange', label=f'Ridge Mean: {np.mean(ridge_mse_scores):.1f}')
plt.xlabel("Fold")
plt.ylabel("MSE")
plt.title("Cross-Validated MSE per Fold")
plt.legend()
plt.grid(True)
plt.show()


<img src="output.png" alt="MSE Comparison of Models" width="600"/>


## Conclusion


- Random Forest shows strong performance with MSE ≈ 306.5 — close to the reported 307.6 in the paper 
- Ridge Regression performs worse, with MSE ≈ 418.5 — expected for a linear model
- These baselines confirm that our data pipeline, folds, and processing are correct
- Next step: Run and compare DeepSynergy under the same setup
